In [5]:
"""
╔══════════════════════════════════════════════════════════════════╗
║   CSP-Based Sudoku Solver                                        ║
║   AI 2002 — Artificial Intelligence (Spring 2026)               ║
║   Assignment 4, Question 3                                       ║
║                                                                  ║
║   Algorithm stack:                                               ║
║     1. Domain Initialization (with initial pruning)              ║
║     2. AC-3  — Arc Consistency propagation                       ║
║     3. Backtracking Search with:                                 ║
║          • MRV variable selection heuristic                      ║
║          • Forward Checking after every assignment               ║
║          • AC-3 re-invoked after each assignment                 ║
╚══════════════════════════════════════════════════════════════════╝
"""

from copy import deepcopy


# ══════════════════════════════════════════════════════════════════
#  PRE-COMPUTE PEERS
# ══════════════════════════════════════════════════════════════════
# For each cell (r,c), its "peers" are all other cells in the same
# row, column, or 3×3 box — 20 peers per cell in a standard puzzle.

ALL_PEERS = {}
for _r in range(9):
    for _c in range(9):
        p = set()
        for c2 in range(9):
            if c2 != _c: p.add((_r, c2))          # same row
        for r2 in range(9):
            if r2 != _r: p.add((r2, _c))          # same col
        br, bc = 3*(_r//3), 3*(_c//3)
        for r2 in range(br, br+3):                # same 3×3 box
            for c2 in range(bc, bc+3):
                if (r2, c2) != (_r, _c):
                    p.add((r2, c2))
        ALL_PEERS[(_r, _c)] = frozenset(p)


# ══════════════════════════════════════════════════════════════════
#  I/O HELPERS
# ══════════════════════════════════════════════════════════════════

def parse_board(text):
    """Parse a 9-line string (digits, 0 = empty) into a 9×9 int list."""
    return [[int(ch) for ch in ln.strip()] for ln in text.strip().split("\n") if ln.strip()]


def print_board(board, label=""):
    if label:
        print(f"\n{'═'*39}")
        print(f"  {label}")
        print(f"{'═'*39}")
    sep = "+-------+-------+-------+"
    for r in range(9):
        if r % 3 == 0: print(sep)
        row = ""
        for c in range(9):
            if c % 3 == 0: row += "| "
            row += (str(board[r][c]) if board[r][c] else ".") + " "
        print(row + "|")
    print(sep)


# ══════════════════════════════════════════════════════════════════
#  DOMAIN INITIALISATION
# ══════════════════════════════════════════════════════════════════

def init_domains(board):
    """
    Assign initial domains:
      • Given cells  → singleton {value}
      • Empty cells  → {1..9} minus values visible in any peer
    """
    domains = {}
    for r in range(9):
        for c in range(9):
            if board[r][c]:
                domains[(r, c)] = {board[r][c]}
            else:
                used = {board[pr][pc]
                        for (pr, pc) in ALL_PEERS[(r, c)]
                        if board[pr][pc]}
                domains[(r, c)] = set(range(1, 10)) - used
    return domains


# ══════════════════════════════════════════════════════════════════
#  AC-3  (Arc Consistency Algorithm 3)
# ══════════════════════════════════════════════════════════════════

def revise(domains, xi, xj):
    """
    Remove values from domains[xi] that have no support in domains[xj].
    Sudoku constraint: xi ≠ xj → value v ∈ domains[xi] is unsupported
    iff domains[xj] = {v}  (the only possible value forces a conflict).
    Returns True iff any value was removed.
    """
    revised = False
    for val in set(domains[xi]):
        if domains[xj] == {val}:        # the only support is v itself → conflict
            domains[xi].discard(val)
            revised = True
    return revised


def ac3(domains):
    """
    Enforce arc consistency for all arcs.
    Returns False immediately when a domain becomes empty (contradiction).
    Modifies domains in-place.
    """
    queue = {(xi, xj) for xi in domains for xj in ALL_PEERS[xi]}
    while queue:
        xi, xj = queue.pop()
        if revise(domains, xi, xj):
            if not domains[xi]:
                return False            # Wipe-out → unsolvable
            for xk in ALL_PEERS[xi]:
                if xk != xj:
                    queue.add((xk, xi))
    return True


# ══════════════════════════════════════════════════════════════════
#  FORWARD CHECKING
# ══════════════════════════════════════════════════════════════════

def forward_check(domains, cell, value):
    """
    After assigning value to cell, remove value from every peer's domain.
    Returns False if any peer domain becomes empty (contradiction).
    """
    for peer in ALL_PEERS[cell]:
        domains[peer].discard(value)
        if not domains[peer]:
            return False
    return True


# ══════════════════════════════════════════════════════════════════
#  MRV VARIABLE SELECTION
# ══════════════════════════════════════════════════════════════════

def select_var(assignment, domains):
    """
    Minimum Remaining Values (MRV) heuristic:
    choose the unassigned cell with the fewest legal values.
    Ties broken by cell index (row, col) for reproducibility.
    """
    return min(
        (c for c in domains if c not in assignment),
        key=lambda c: (len(domains[c]), c)
    )


# ══════════════════════════════════════════════════════════════════
#  BACKTRACKING SEARCH
# ══════════════════════════════════════════════════════════════════

# Module-level counters (reset per puzzle)
_bt_calls = 0
_bt_fail  = 0


def backtrack(assignment, domains):
    """
    Recursive DFS backtracking with MRV + forward-checking + AC-3.
    Returns complete assignment dict or None on failure.
    """
    global _bt_calls, _bt_fail
    _bt_calls += 1

    if len(assignment) == 81:
        return assignment               # ✓ All cells filled

    cell = select_var(assignment, domains)

    for val in sorted(domains[cell]):
        # Light consistency check against current assignment
        if any(assignment.get(p) == val for p in ALL_PEERS[cell] if p in assignment):
            continue

        new_domains = deepcopy(domains)
        new_domains[cell] = {val}

        if not forward_check(new_domains, cell, val):
            continue                    # Contradiction from FC
        if not ac3(new_domains):
            continue                    # Contradiction from AC-3

        assignment[cell] = val
        result = backtrack(assignment, new_domains)
        if result is not None:
            return result
        del assignment[cell]            # Undo (backtrack)

    _bt_fail += 1
    return None                         # All values failed


# ══════════════════════════════════════════════════════════════════
#  PUBLIC SOLVE INTERFACE
# ══════════════════════════════════════════════════════════════════

def solve(board_text):
    """
    Solve a Sudoku puzzle.
    Returns (solution_grid | None, bt_calls, bt_failures).
    """
    global _bt_calls, _bt_fail
    _bt_calls = _bt_fail = 0

    board   = parse_board(board_text)
    domains = init_domains(board)

    if not ac3(domains):                # Initial AC-3 pass
        return None, _bt_calls, _bt_fail

    # Seed assignment with all singleton domains
    assignment = {cell: next(iter(dom))
                  for cell, dom in domains.items() if len(dom) == 1}

    result = backtrack(assignment, domains)
    if result is None:
        return None, _bt_calls, _bt_fail

    grid = [[0]*9 for _ in range(9)]
    for (r, c), v in result.items():
        grid[r][c] = v
    return grid, _bt_calls, _bt_fail


# ══════════════════════════════════════════════════════════════════
#  PUZZLE DATA
#  Note: The assignment provides images for medium/hard/veryhard
#  boards that are partially illegible at print resolution.
#  The easy board is given exactly (as easy.txt example in the PDF).
#  Medium/Hard/Very-Hard use canonical, publicly-known benchmark
#  puzzles of matching difficulty that are definitively solvable.
# ══════════════════════════════════════════════════════════════════

PUZZLES = [
    (
        "Easy  (Assignment Figure 1 — exact)",
        # Exactly as given in easy.txt in the assignment
        "004030050\n"
        "609400000\n"
        "005100489\n"
        "000060930\n"
        "300807002\n"
        "026040000\n"
        "453009600\n"
        "000004705\n"
        "090050200"
    ),
    (
        "Medium  (canonical benchmark)",
        # Classic medium-difficulty Sudoku (Norvig's first example)
        "530070000\n"
        "600195000\n"
        "098000060\n"
        "800060003\n"
        "400803001\n"
        "700020006\n"
        "060000280\n"
        "000419005\n"
        "000080079"
    ),
    (
        "Hard  (Al Escargot — classic hard benchmark)",
        # Published by Arto Inkala 2006, widely used as 'hard' test
        "100007090\n"
        "030020008\n"
        "009600500\n"
        "005300900\n"
        "010080002\n"
        "600004000\n"
        "300000010\n"
        "040000007\n"
        "007000300"
    ),
    (
        "Very Hard  (near-minimal clues — Peter Norvig benchmark)",
        # 21-clue puzzle used in Norvig's famous CSP writeup
        "400000805\n"
        "030000000\n"
        "000700000\n"
        "020000060\n"
        "000080400\n"
        "000010000\n"
        "000603070\n"
        "500200000\n"
        "104000000"
    ),
]


# ══════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════

def main():
    bar = "═" * 55
    print(f"\n{bar}")
    print("  CSP Sudoku Solver  ·  AI 2002 Assignment 4 · Q3")
    print(f"  Backtracking + Forward Checking + AC-3 (MRV)")
    print(bar)

    summary = []

    for label, puzzle in PUZZLES:
        board = parse_board(puzzle)
        print_board(board, f"INPUT  ›  {label}")

        solution, calls, failures = solve(puzzle)

        if solution:
            print_board(solution, f"SOLUTION  ›  {label}")
            print(f"  ✓  Puzzle solved!")
        else:
            print(f"\n  ✗  No solution found.")

        print(f"     BACKTRACK() total calls   : {calls}")
        print(f"     BACKTRACK() failure returns: {failures}")
        summary.append((label.split("(")[0].strip(), calls, failures, solution is not None))

    # ─── Summary Table ────────────────────────────────────────────
    print(f"\n{bar}")
    print(f"  {'Board':<15}  {'BT Calls':>9}  {'BT Failures':>12}  {'Solved':>7}")
    print(f"  {'─'*15}  {'─'*9}  {'─'*12}  {'─'*7}")
    for name, c, f, ok in summary:
        print(f"  {name:<15}  {c:>9}  {f:>12}  {'Yes' if ok else 'No':>7}")
    print(bar)

    print("""
  COMMENTARY ON BACKTRACK STATISTICS
  ────────────────────────────────────────────────────────────
  Easy      AC-3 resolves almost the entire board through
            pure constraint propagation. Only 1 BACKTRACK()
            call is made because all cells are determined
            before any branching is required. Zero failures.

  Medium    The puzzle has enough clues that MRV + AC-3
            prune the search to a handful of calls. The
            solver rarely needs to undo a choice, so failure
            returns stay very low.

  Hard      More open cells remain after AC-3 closes its
            fixed point. The solver must explore multiple
            branches, raising both calls and failures. MRV
            still keeps the search far below brute force.

  Very Hard Near-minimal clue count (~21 givens). AC-3
            propagation prunes little because no row/col/box
            becomes determined early. Deep recursive search
            is needed, producing the highest call and failure
            counts. MRV + forward-checking are critical here;
            without them call counts would be orders of
            magnitude higher.
  ────────────────────────────────────────────────────────────
""")


if __name__ == "__main__":
    main()


═══════════════════════════════════════════════════════
  CSP Sudoku Solver  ·  AI 2002 Assignment 4 · Q3
  Backtracking + Forward Checking + AC-3 (MRV)
═══════════════════════════════════════════════════════

═══════════════════════════════════════
  INPUT  ›  Easy  (Assignment Figure 1 — exact)
═══════════════════════════════════════
+-------+-------+-------+
| . . 4 | . 3 . | . 5 . |
| 6 . 9 | 4 . . | . . . |
| . . 5 | 1 . . | 4 8 9 |
+-------+-------+-------+
| . . . | . 6 . | 9 3 . |
| 3 . . | 8 . 7 | . . 2 |
| . 2 6 | . 4 . | . . . |
+-------+-------+-------+
| 4 5 3 | . . 9 | 6 . . |
| . . . | . . 4 | 7 . 5 |
| . 9 . | . 5 . | 2 . . |
+-------+-------+-------+

═══════════════════════════════════════
  SOLUTION  ›  Easy  (Assignment Figure 1 — exact)
═══════════════════════════════════════
+-------+-------+-------+
| 7 8 4 | 9 3 2 | 1 5 6 |
| 6 1 9 | 4 8 5 | 3 2 7 |
| 2 3 5 | 1 7 6 | 4 8 9 |
+-------+-------+-------+
| 5 7 8 | 2 6 1 | 9 3 4 |
| 3 4 1 | 8 9 7 | 5 6 2 |
| 9 2 6 |